
## drop (delete)
- 0  customer id = drop

## Convert
- 19  TotalCharges     object > float ,   this creates 11 NaNs →  i fill them with 0
##### TotalCharges NaNs:
##### When i convert TotalCharges to float, 11 rows become empty (NaN).
##### These are new customers (tenure=0) who haven't been billed yet.
##### need to Fill them with 0.
  
## stays (don't touch)
- 2  SeniorCitizen = ok
- 5  tenture = ok
- 18  MonthlyCharges = ok
  
## Binary encoding
- 1  gender      male = 1, famale = 0
- 3  Partner             yes = 1, no = 0
- 4  Depandants          yes = 1, no = 0
- 6  phone service       yes = 1, no = 0
- 16  PaperlessBilling  yes = 1, no = 0
- 20  Churn             yes = 1, no = 0
  
## ordinal 
 15  Contract         Ordinal map
 Ordinal: Contract (Month-to-month=0, One year=1, Two year=2)

## One hot
- 7  MultipleLines =   one hot
- 8  InternetService   one hot
- 9  OnlineSecurity    one hot
- 10  OnlineBackup     one hot  
- 11  DeviceProtection one hot
- 12  TechSupport      one hot 
- 13  StreamingTV      one hot 
- 14  StreamingMovies  one hot 
- 17  PaymentMethod    one hot
#### (The service columns will create some duplicate "No internet service" columns. it is okay)

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

In [2]:
# getting docs
raw_data = pd.read_csv('/Users/shavkatjon/telco-c-classifier/data/raw/telco_churn.csv')

In [3]:
#working copy
df = raw_data

In [4]:
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


# 1 data cleaning

### 1.1 droping customer id 

In [5]:
# Drop the customerID column as it holds no predictive power for the model
df = df.drop('customerID', axis=1)

In [6]:
print(df.shape)

(7043, 20)


### 1.2 Total charges

In [7]:
# Phase 1: Convert TotalCharges 
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# VERIFICATION: Check how many NaNs were 
print("Missing values after conversion:", df['TotalCharges'].isna().sum())

# Phase 2: Fill those 11 missing values with 0
df['TotalCharges'] = df['TotalCharges'].fillna(0)

Missing values after conversion: 11


In [8]:
# verification
print("Missing values after fill:", df['TotalCharges'].isna().sum())
print("Current Data Type:", df['TotalCharges'].dtype)


Missing values after fill: 0
Current Data Type: float64


In [9]:
# 1. Verify dimensions (Expected output: (7043, 19))
print("DataFrame Shape:", df.shape)

# 2. Confirm customerID is removed (Expected output: False)
print("Is customerID present?:", 'customerID' in df.columns)

# 3. Verify TotalCharges data type (Expected output: float64)
print("TotalCharges Data Type:", df['TotalCharges'].dtype)

# 4. Count total missing values across ALL columns (Expected output: 0)
print("Total Missing Values in DataFrame:", df.isna().sum().sum())

DataFrame Shape: (7043, 20)
Is customerID present?: False
TotalCharges Data Type: float64
Total Missing Values in DataFrame: 0


# 2 encoding  
## 2.1 binary

In [10]:
# mapping gender 
df['gender'] = df['gender'].map({'Female': 0, 'Male': 1})

#defining standart bin dictionary
binary_map = {'No': 0, 'Yes': 1}

In [11]:
# apply to others 
binary_cols = ['Partner', 'Dependents', 'PhoneService', 'PaperlessBilling','Churn']

for col in binary_cols:
    df[col] = df[col].map(binary_map)

In [12]:
#verify
print(df[['gender'] + binary_cols].head())
print("\nData types of converted columns:")
print(df[['gender'] + binary_cols].dtypes)

   gender  Partner  Dependents  PhoneService  PaperlessBilling  Churn
0       0        1           0             0                 1      0
1       1        0           0             1                 0      0
2       1        0           0             1                 1      1
3       1        0           0             0                 0      0
4       0        0           0             1                 1      1

Data types of converted columns:
gender              int64
Partner             int64
Dependents          int64
PhoneService        int64
PaperlessBilling    int64
Churn               int64
dtype: object


In [13]:
# explicit hierarchy mapping for contract duration
contract_mapping = {
    'Month-to-month': 0,
    'One year': 1,
    'Two year': 2
}

# Map the string values in the Contract column to integers
df['Contract'] = df['Contract'].map(contract_mapping)

# VERIFICATION:
print(df['Contract'].value_counts(dropna=False))
print("\nContract Data Type:", df['Contract'].dtype)

Contract
0    3875
2    1695
1    1473
Name: count, dtype: int64

Contract Data Type: int64


## 2.2 One-hot encoding

In [14]:
# initialize ohe (standart Numpy array)
ohe = OneHotEncoder(sparse_output=False)

#Fit transform it 
payment_encoded = ohe.fit_transform(df[['PaymentMethod']])

#categories learned by the encoder
print("Learned Categories:", ohe.categories_)

# view new column names from onehot 
print("\nNew Feature Names:", ohe.get_feature_names_out(['PaymentMethod']))

# see the matrix output
print("\nTransformed Binary Matrix (First 5 rows):\n", payment_encoded[:5])

Learned Categories: [array(['Bank transfer (automatic)', 'Credit card (automatic)',
       'Electronic check', 'Mailed check'], dtype=object)]

New Feature Names: ['PaymentMethod_Bank transfer (automatic)'
 'PaymentMethod_Credit card (automatic)' 'PaymentMethod_Electronic check'
 'PaymentMethod_Mailed check']

Transformed Binary Matrix (First 5 rows):
 [[0. 0. 1. 0.]
 [0. 0. 0. 1.]
 [0. 0. 0. 1.]
 [1. 0. 0. 0.]
 [0. 0. 1. 0.]]


## 2.3 Nominal Encoding

In [15]:
# Define all nominal feature column names
nominal_cols = [
    'MultipleLines', 'InternetService', 'OnlineSecurity',
    'OnlineBackup', 'DeviceProtection', 'TechSupport',
    'StreamingTV', 'StreamingMovies', 'PaymentMethod'
]

# Initialize OneHotEncoder (sparse_output=False returns a standard NumPy matrix)
ohe = OneHotEncoder(sparse_output=False)

# Fit and transform all nominal columns simultaneously
encoded_array = ohe.fit_transform(df[nominal_cols])

In [16]:
# Extract generated column names (e.g., 'InternetService_Fiber optic')
encoded_col_names = ohe.get_feature_names_out(nominal_cols)

# Create a temporary DataFrame with matching row indices
df_encoded = pd.DataFrame(encoded_array, columns=encoded_col_names, index=df.index)

# Drop original text columns from 'df' and join the newly encoded binary columns
df = df.drop(nominal_cols, axis=1).join(df_encoded)

In [17]:
# VERIFICATION: Check the expanded column dimensions (Expect ~40+ features)
print("Updated DataFrame Shape:", df.shape)
print("\nFirst 5 Column Names:", df.columns[:5].tolist())

Updated DataFrame Shape: (7043, 39)

First 5 Column Names: ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure']


#### checking

In [18]:
# 1. Verify updated shape (Expect 7043 rows and ~40+ columns)
print("Updated DataFrame Shape:", df.shape)

# 2. Check summary count of column data types (Should show ONLY float64 or int64)
print("\nData Types Summary:")
print(df.dtypes.value_counts())

# 3. Confirm zero missing values
print("\nTotal Missing Values in DataFrame:", df.isna().sum().sum())

# 4. Display sample rows to verify all features are strictly numeric
df.head()

Updated DataFrame Shape: (7043, 39)

Data Types Summary:
float64    30
int64       9
Name: count, dtype: int64

Total Missing Values in DataFrame: 0


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,Contract,PaperlessBilling,MonthlyCharges,TotalCharges,...,StreamingTV_No,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No,StreamingMovies_No internet service,StreamingMovies_Yes,PaymentMethod_Bank transfer (automatic),PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0,0,1,0,1,0,0,1,29.85,29.85,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
1,1,0,0,0,34,1,1,0,56.95,1889.50,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0
2,1,0,0,0,2,1,0,1,53.85,108.15,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0
3,1,0,0,0,45,0,1,0,42.30,1840.75,...,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0
4,0,0,0,0,2,1,0,1,70.70,151.65,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0


In [19]:
# this is complete 
# now time to train and test model 